# Qwen Agentic Fine-Tune (Google Colab Free)

Fine-tune **Qwen3.5-2B-Instruct** with Unsloth LoRA using this repo.

## Before you start
1. **Runtime → Change runtime type → T4 GPU** (free tier).
2. Upload `train.jsonl` + `val.jsonl` from your PC (`AI-Exploration/data/processed/`) into Google Drive, **or** use the upload cell.
3. Run cells **top to bottom**.

## Recommended flow on free Colab
1. Smoke test: `MAX_STEPS = 10`
2. If that works, raise to `200` / `400`, then set `MAX_STEPS = None` for full epochs
3. Download the LoRA adapter zip at the end

## 0) Config — edit these first

In [ ]:
# GitHub repo that contains the training code
REPO_URL = "https://github.com/Gilga1/AI-Exploration.git"
REPO_BRANCH = "cursor/qwen-agentic-ft-setup-7e0b"  # change if your branch differs
REPO_DIR = "/content/AI-Exploration"

# Where to keep outputs so disconnects don't wipe them
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/qwen-agentic-ft"

# Training knobs for Colab free (T4)
MAX_STEPS = 10          # set to None for full epochs from training.yaml
USE_4BIT = True         # safer VRAM on T4 (~15GB)
SAVE_MERGED = False     # skip merge on free tier; download LoRA adapter only
BATCH_SIZE = 1          # raise to 2 if VRAM allows
GRAD_ACCUM = 16         # keep effective batch ~16

# How to get training data into Colab
# "drive"   = copy from Google Drive folder below
# "upload"  = file picker in this notebook
# "extract" = clone GitHub code repos and rebuild JSONL (slow)
DATA_SOURCE = "drive"
DRIVE_DATA_DIR = f"{DRIVE_PROJECT_DIR}/data/processed"  # put train.jsonl + val.jsonl here

## 1) Check GPU (must be NVIDIA)

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU. Runtime → Change runtime type → T4 GPU, then Runtime → Restart session."
    )
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 2) Mount Google Drive (recommended)

In [ ]:
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    Path(DRIVE_PROJECT_DIR).mkdir(parents=True, exist_ok=True)
    Path(DRIVE_DATA_DIR).mkdir(parents=True, exist_ok=True)
    print("Drive project:", DRIVE_PROJECT_DIR)
    print("Put train.jsonl + val.jsonl in:", DRIVE_DATA_DIR)
else:
    print("Skipping Drive mount.")

## 3) Clone repo + install Unsloth / project

In [ ]:
import os
import subprocess
from pathlib import Path

def sh(cmd: str):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if Path(REPO_DIR).exists():
    print("Repo already present:", REPO_DIR)
else:
    sh(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())

In [ ]:
%%capture
# Unsloth Colab install (latest from GitHub) + project deps
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q -e ".[train]"

In [ ]:
from unsloth import FastLanguageModel
print("Unsloth OK:", FastLanguageModel)

## 4) Load training data

Your local files live at `AI-Exploration/data/processed/train.jsonl` (~17 MB) and `val.jsonl`.

**Easiest path:** upload both into Google Drive folder  
`MyDrive/qwen-agentic-ft/data/processed/` then keep `DATA_SOURCE = "drive"` above.

In [ ]:
import shutil
from pathlib import Path

processed = Path(REPO_DIR) / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)
train_dst = processed / "train.jsonl"
val_dst = processed / "val.jsonl"

if DATA_SOURCE == "drive":
    src_train = Path(DRIVE_DATA_DIR) / "train.jsonl"
    src_val = Path(DRIVE_DATA_DIR) / "val.jsonl"
    if not src_train.exists():
        raise FileNotFoundError(
            f"Missing {src_train}. Upload train.jsonl (and ideally val.jsonl) to Drive first."
        )
    shutil.copy2(src_train, train_dst)
    if src_val.exists():
        shutil.copy2(src_val, val_dst)
    print("Copied data from Drive.")

elif DATA_SOURCE == "upload":
    from google.colab import files

    print("Select train.jsonl and optionally val.jsonl...")
    uploaded = files.upload()
    for name, blob in uploaded.items():
        target = processed / Path(name).name
        target.write_bytes(blob)
        print("Saved", target)

elif DATA_SOURCE == "extract":
    print("Extracting from GitHub repos (can take a while)...")
    sh("python scripts/extract_data.py")

else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

if not train_dst.exists():
    raise FileNotFoundError(f"Training file missing: {train_dst}")

n_train = sum(1 for _ in train_dst.open(encoding="utf-8") if _.strip())
n_val = sum(1 for _ in val_dst.open(encoding="utf-8") if _.strip()) if val_dst.exists() else 0
print(f"train examples: {n_train}")
print(f"val examples:   {n_val}")
print(f"train size MB:  {train_dst.stat().st_size / 1e6:.2f}")

## 5) Apply Colab-friendly overrides + train

In [ ]:
import json
from pathlib import Path

from qwen_agentic_ft.config import ROOT
from qwen_agentic_ft.train.data import load_training_config
from qwen_agentic_ft.train.sft import run_training

config = load_training_config()

# Colab free / T4 overrides
config["load_in_4bit"] = USE_4BIT
config["load_in_16bit"] = not USE_4BIT
config["training"]["bf16"] = False   # T4: use fp16
config["training"]["fp16"] = True
config["training"]["per_device_train_batch_size"] = BATCH_SIZE
config["training"]["per_device_eval_batch_size"] = BATCH_SIZE
config["training"]["gradient_accumulation_steps"] = GRAD_ACCUM
config["export"]["save_merged"] = SAVE_MERGED

# Absolute Drive paths survive Colab disconnects
# (pathlib ignores ROOT when the right-hand path is absolute on Linux)
if USE_GOOGLE_DRIVE:
    config["training"]["output_dir"] = str(Path(DRIVE_PROJECT_DIR) / "outputs" / "qwen-agentic-lora")
    config["export"]["merged_dir"] = str(Path(DRIVE_PROJECT_DIR) / "outputs" / "qwen-agentic-merged")

if MAX_STEPS is not None:
    config["training"]["max_steps"] = int(MAX_STEPS)
    config["training"]["num_train_epochs"] = 1

print(json.dumps({
    "model_name": config["model_name"],
    "load_in_4bit": config["load_in_4bit"],
    "bf16": config["training"]["bf16"],
    "fp16": config["training"]["fp16"],
    "batch": config["training"]["per_device_train_batch_size"],
    "grad_accum": config["training"]["gradient_accumulation_steps"],
    "max_steps": config["training"].get("max_steps"),
    "epochs": config["training"]["num_train_epochs"],
    "output_dir": config["training"]["output_dir"],
    "save_merged": config["export"]["save_merged"],
}, indent=2))

In [ ]:
result = run_training(config, ROOT)
print(json.dumps(result, indent=2, default=str))

## 6) Zip + download LoRA adapter

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

final_dir = Path(result["output_dir"]) / "final"
if not final_dir.exists():
    raise FileNotFoundError(f"Expected adapter at {final_dir}")

zip_base = Path("/content/qwen-agentic-lora-final")
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=final_dir))
print("Created", zip_path, "size MB:", round(zip_path.stat().st_size / 1e6, 2))

if USE_GOOGLE_DRIVE:
    drive_zip = Path(DRIVE_PROJECT_DIR) / zip_path.name
    shutil.copy2(zip_path, drive_zip)
    print("Also saved to Drive:", drive_zip)

files.download(str(zip_path))

## Tips if something fails

| Symptom | Fix |
|---|---|
| `CUDA available: False` | Runtime → T4 GPU, then restart |
| Unsloth / NVIDIA-only error | Same as above |
| CUDA OOM | Keep `USE_4BIT=True`, `BATCH_SIZE=1`, or lower `max_seq_length` in `config/training.yaml` |
| Session disconnected | Re-run from top; checkpoints on Drive under `qwen-agentic-ft/outputs/` |
| Full 2-epoch run too long | Raise `MAX_STEPS` gradually (`50` → `200` → `None`) or use Colab Pro / RunPod |
| Git clone branch not found | Set `REPO_BRANCH` to `main` (or your branch) in the config cell |

After a successful smoke test (`MAX_STEPS = 10`), set `MAX_STEPS = None` and re-run the training cells for a longer job.